# 01 — Perfil dos dados

Regenera `docs/relatorio_analise_dados.md` sobre a série inteira.

Duas diferenças em relação à versão anterior deste notebook. Ele lia o **DuckDB
intermediário**, onde tudo é VARCHAR, e cobria apenas 201701 e 202501. Agora lê a
**camada primária** — Parquet tipado, só com as colunas declaradas em
`docs/01-selecao-tabelas.md` — e cobre os nove snapshots.

A consequência prática: as estatísticas passam a descrever o dado que os modelos
realmente veem. Perfilar VARCHAR antes da conversão media coisas como
`qt_existente` sem saber que é inteiro, e mede nulos antes de o `TRY_CAST` anular
o que não converte.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import pandas as pd

from src import changes, schema
from src.paths import DOCS_DIR, PRIMARY_FOLDER

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

con = duckdb.connect()
PERIODOS = changes.periodos_disponiveis()
print(f"snapshots: {PERIODOS}")
print(f"tabelas no escopo: {len(schema.FACT_TABLES)}")
assert PERIODOS, "rode `python -m src.pipeline` antes"


## Perfil de uma tabela

Por coluna: percentual de nulos, percentual de valores distintos, moda e sua
frequência. O percentual de distintos é o que separa identificador de categoria —
próximo de 100% é chave, próximo de zero é categoria de baixa cardinalidade.

In [ ]:
def perfilar(tabela: str, periodo: str) -> tuple[pd.DataFrame | None, int]:
    caminho = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
    if not caminho.exists():
        return None, 0

    colunas = [r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{caminho}')").fetchall()]
    total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{caminho}')").fetchone()[0]
    if total == 0:
        return None, 0

    # Uma query por tabela, não uma por coluna: 44 tabelas x 9 snapshots x ~9
    # colunas seria mais de 3500 varreduras do mesmo arquivo.
    agregados = ", ".join(
        f'COUNT("{c}") AS nn_{i}, COUNT(DISTINCT "{c}") AS nd_{i}'
        for i, c in enumerate(colunas)
    )
    r = con.execute(f"SELECT {agregados} FROM read_parquet('{caminho}')").df().iloc[0]

    linhas = []
    for i, coluna in enumerate(colunas):
        moda = con.execute(f'''
            SELECT "{coluna}" AS v, COUNT(*) AS n FROM read_parquet('{caminho}')
            WHERE "{coluna}" IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 1
        ''').fetchall()
        linhas.append({
            "coluna": coluna,
            "%_nulos": round(100 * (total - r[f"nn_{i}"]) / total, 4),
            "%_distintos": round(100 * r[f"nd_{i}"] / total, 4),
            "moda": str(moda[0][0])[:40] if moda else None,
            "%_moda": round(100 * moda[0][1] / total, 4) if moda else 0.0,
        })
    return pd.DataFrame(linhas), total


df, n = perfilar("rlEstabEquipamento", PERIODOS[-1])
print(f"rlEstabEquipamento em {PERIODOS[-1]}: {n:,} linhas\n")
print(df.to_string(index=False))


## Evolução de uma coluna ao longo da série

Onde o filtro empírico de D-06 age. Uma coluna que muda de comportamento entre
snapshots é mais perigosa que uma constantemente ruim, porque o modelo aprende um
regime e é avaliado em outro.

In [ ]:
def evolucao(tabela: str, coluna: str) -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        caminho = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        if not caminho.exists():
            continue
        cols = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{caminho}')").fetchall()}
        if coluna not in cols:
            linhas.append({"periodo": periodo, "obs": "coluna ausente"})
            continue
        r = con.execute(f'''
            SELECT COUNT(*) n, COUNT("{coluna}") nn, COUNT(DISTINCT "{coluna}") nd
            FROM read_parquet('{caminho}')
        ''').fetchone()
        linhas.append({"periodo": periodo, "linhas": r[0],
                       "%_nulos": round(100 * (r[0] - r[1]) / r[0], 2) if r[0] else None,
                       "distintos": r[2]})
    return pd.DataFrame(linhas)

for coluna in ("qt_existente", "tp_sus", "to_chardt_atualizacao_origemddmmyyyy"):
    print(f"\nrlEstabEquipamento.{coluna}")
    print(evolucao("rlEstabEquipamento", coluna).to_string(index=False))


## Regenerar o relatório

Escreve `docs/relatorio_analise_dados.md` cobrindo todas as tabelas do escopo em
todos os snapshots. É insumo do filtro empírico e do notebook 00.

In [ ]:
def gerar_relatorio(destino: Path | None = None, periodos: list[str] | None = None) -> Path:
    destino = destino or DOCS_DIR / "relatorio_analise_dados.md"
    periodos = periodos or PERIODOS

    partes = [
        "# Relatório de análise do CNES\n\n",
        "Gerado por `notebook/01_perfil_dados.ipynb` a partir da camada primária\n",
        "(`data/03_primary`), portanto sobre o dado já tipado e restrito às colunas\n",
        "declaradas em [`01-selecao-tabelas.md`](01-selecao-tabelas.md).\n\n",
        f"Competências: {', '.join(periodos)}.\n\n",
    ]
    for tabela in sorted(schema.FACT_TABLES):
        for periodo in periodos:
            df, total = perfilar(tabela, periodo)
            partes.append(f"### {tabela} (competência: {periodo})\n\n")
            if df is None:
                partes.append("Tabela ausente ou vazia neste snapshot.\n\n")
                continue
            partes.append(f"**Linhas:** {total:,}\n\n".replace(",", "."))
            partes.append(df.to_markdown(index=False) + "\n\n")

    destino.write_text("".join(partes), encoding="utf-8")
    return destino

# Descomente para regenerar (leva alguns minutos: 44 tabelas x 9 snapshots).
# caminho = gerar_relatorio()
# print(f"escrito: {caminho} ({caminho.stat().st_size / 1024:.0f} KB)")
